# GemmaMicro — Treino PT-BR no Colab

**Ordem de execução:** rode as células de cima para baixo.

- GPU recomendada: T4 (gratuita) ou A100 (Colab Pro)
- Salva automaticamente no Google Drive
- Tempo estimado: ~2h (T4) / ~40min (A100)

## 1. Verifica GPU e instala dependências

In [1]:
import subprocess, sys

# Checa GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NENHUMA — vá em Runtime > Change runtime type > T4 GPU')

# Instala dependências
!pip install -q torch tokenizers datasets trafilatura requests
print('Dependências instaladas.')

GPU: Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 6.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 34.2 MB/s eta 0:00:00
Dependências instaladas.


## 2. Monta Google Drive (salva modelo aqui)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Testa escrita
_t = f'{DRIVE_DIR}/_test.txt'
open(_t, 'w').write('ok')
assert os.path.exists(_t)
os.remove(_t)
print(f'Drive OK → {DRIVE_DIR}')

Mounted at /content/drive
Drive OK → /content/drive/MyDrive/GemmaMicro


## 3. Clona o repositório

In [5]:
import os

REPO = '/content/IA'
if not os.path.exists(REPO):
    !git clone https://github.com/viniciusdev772/IA.git {REPO}
else:
    !git -C {REPO} pull

print('Repo pronto.')

Cloning into '/content/IA'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 40 (delta 6), reused 40 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 484.30 KiB | 2.07 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Repo pronto.


## 4. Baixa datasets do HuggingFace

Baixa Wikipedia PT-BR completa + mc4 (Common Crawl PT-BR).  
Pode levar 10-20 min dependendo da conexão.

In [ ]:
import os, re
from datasets import load_dataset

DATA_DIR = '/content/datasets'
os.makedirs(DATA_DIR, exist_ok=True)

def limpar_texto(txt: str) -> str:
    txt = re.sub(r'\[\d+\]', '', txt)
    txt = re.sub(r'=+[^=]+=+', '', txt)
    txt = re.sub(r'[ \t]+', ' ', txt)
    return txt.strip()

def salvar_dataset(nome_arquivo, textos, min_chars=80):
    path = f'{DATA_DIR}/{nome_arquivo}'
    if os.path.exists(path):
        print(f'  {nome_arquivo} já existe, pulando.')
        return path
    count = 0
    with open(path, 'w', encoding='utf-8') as f:
        for txt in textos:
            txt = limpar_texto(txt)
            if len(txt) >= min_chars:
                f.write(txt + '\n')
                count += 1
    print(f'  {nome_arquivo}: {count:,} frases salvas')
    return path

# ── Wikipedia PT-BR (formato Parquet novo) ─────────────────────────────────
print('Baixando Wikipedia PT-BR (wikimedia/wikipedia)...')
wiki = load_dataset(
    'wikimedia/wikipedia',
    '20231101.pt',
    split='train',
)
wiki_path = salvar_dataset(
    'wiki_pt.txt',
    (row['text'] for row in wiki)
)
del wiki
print(f'Wikipedia salva: {os.path.getsize(wiki_path)/1024/1024:.1f} MB')

# ── CulturaX PT (Common Crawl filtrado, substitui mc4) ─────────────────────
print('\nBaixando CulturaX PT-BR (50k amostras, streaming)...')
try:
    cx = load_dataset(
        'uonlp/CulturaX',
        'pt',
        split='train',
        streaming=True,
    )
    cx_path = salvar_dataset(
        'culturax_pt.txt',
        (row['text'] for i, row in enumerate(cx) if i < 50_000)
    )
    del cx
    print(f'CulturaX salvo: {os.path.getsize(cx_path)/1024/1024:.1f} MB')
except Exception as e:
    print(f'CulturaX falhou ({e}), continuando sem ele.')
    cx_path = None

print('\nDownload concluído!')


## 5. Lista todos os datasets disponíveis

In [ ]:
import os, glob

REPO = '/content/IA'
DATA_DIR = '/content/datasets'

# Datasets do repo (pequenos, já commitados)
repo_datasets = [
    f'{REPO}/dataset_portugues_br.txt',
    f'{REPO}/frases_treinamento.txt',
    f'{REPO}/dataset_v2.txt',
    f'{REPO}/dataset_v3.txt',
    f'{REPO}/dataset_wiki_pt.txt',
]

# Datasets baixados
downloaded = glob.glob(f'{DATA_DIR}/*.txt')

all_paths = [p for p in repo_datasets + downloaded if os.path.exists(p)]

print(f'Datasets encontrados ({len(all_paths)}):')
total_mb = 0
for p in all_paths:
    mb = os.path.getsize(p) / 1024 / 1024
    total_mb += mb
    print(f'  {os.path.basename(p):40s} {mb:7.1f} MB')
print(f'  {"TOTAL":40s} {total_mb:7.1f} MB')

## 6. Treina o modelo GemmaMicro

Parâmetros ajustáveis:
- `epochs` — mais épocas = mais tempo, melhor qualidade
- `ctx_len` — contexto. 512 usa mais VRAM.
- `batch_size` — reduza se der OOM (Out of Memory)

In [ ]:
import sys
sys.path.insert(0, '/content/IA')
import nn

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

nn.train(
    data_paths=all_paths,
    save_dir=DRIVE_DIR,
    epochs=20,
    batch_size=64,       # reduza para 32 se der OOM
    lr=6e-4,
    ctx_len=512,
    accum_steps=4,       # batch efetivo = 64*4 = 256
)

## 7. Testa inferência

In [ ]:
import torch, sys
sys.path.insert(0, '/content/IA')
from nn import GemmaMicro, BPETokenizer

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

tokenizer = BPETokenizer.load(f'{DRIVE_DIR}/tokenizer.json')
ckpt = torch.load(f'{DRIVE_DIR}/model.pt', map_location='cpu', weights_only=True)
model = GemmaMicro(vocab_size=tokenizer.vocab_size)
model.load_state_dict(ckpt)
model.eval()
print(f'Modelo: {sum(p.numel() for p in model.parameters()):,} parâmetros')

def gerar(prompt, max_tokens=120, temp=0.8, top_k=50):
    ids = tokenizer.encode(prompt, add_special=False)
    ids = torch.tensor([ids])
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(ids[:, -512:])
            logit = logits[0, -1, :] / temp
            v, _ = torch.topk(logit, top_k)
            logit[logit < v[-1]] = float('-inf')
            next_id = torch.multinomial(torch.softmax(logit, -1), 1)
            ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)
            if next_id.item() == 258:  # <eos>
                break
    return tokenizer.decode(ids[0].tolist(), skip_special=False)

prompts = [
    'a capital do brasil é',
    'a inteligência artificial',
    'o brasil é um país',
    'a língua portuguesa',
]
for p in prompts:
    print(f'\n→ {p}')
    print(gerar(p))

## 8. (Opcional) Baixa modelo do Drive para /content

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
LOCAL_DIR = '/content/gemma_micro'
os.makedirs(LOCAL_DIR, exist_ok=True)

for f in ['model.pt', 'tokenizer.json']:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{LOCAL_DIR}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        mb = os.path.getsize(dst)/1024/1024
        print(f'Copiado {f}: {mb:.1f} MB')

print('Pronto. Use load_model(\'gemma_micro\') localmente.')